# 面试问题：Agent 可观测性怎样支持 Trace 重放、成本归因和故障定位？

可以直接复述的回答是：第一，每次用户请求有唯一 trace_id，每个模型或工具步骤有 span_id 和 parent_id。第二，Span 要记录开始结束、状态、输入摘要、token、费用和重试。第三，事件必须足以重建执行树，而不是只有不可关联的日志文本。第四，先校验孤儿、重复身份和时间边界，再做重放。第五，成本应归因到具体步骤和请求。第六，用每请求延迟、token、工具费用、失败点和关键路径验证可观测性。下面用差旅助手的五条离线运行实现。

## 真实案例：差旅 Agent 的规划、搜索与预订轨迹

五条脱敏请求覆盖查高铁、查酒店、改签、预算比较和取消预订。每条运行包含 root、LLM plan、tool call 和 answer span，部分运行含重试或失败。时延、token 和费用均为确定性教学数据，不代表真实模型价格或线上 SLA。

In [1]:
runs = [  # 定义五条具有成本和故障语义的差旅请求
    {"trace_id": "T-801", "request": "查询明早上海到杭州高铁", "prompt_tokens": 420, "completion_tokens": 95, "tool": "train_search", "tool_ms": 180, "tool_cost": 0.01, "retries": 0, "status": "ok"},  # 正常高铁查询
    {"trace_id": "T-802", "request": "预订北京两晚 600 元内酒店", "prompt_tokens": 680, "completion_tokens": 140, "tool": "hotel_search", "tool_ms": 360, "tool_cost": 0.03, "retries": 1, "status": "ok"},  # 酒店搜索发生一次重试
    {"trace_id": "T-803", "request": "把 G12 改签到下午", "prompt_tokens": 510, "completion_tokens": 120, "tool": "ticket_change", "tool_ms": 240, "tool_cost": 0.05, "retries": 0, "status": "ok"},  # 改签操作具有外部费用
    {"trace_id": "T-804", "request": "比较飞机和高铁总预算", "prompt_tokens": 920, "completion_tokens": 210, "tool": "multi_search", "tool_ms": 510, "tool_cost": 0.08, "retries": 2, "status": "ok"},  # 多源比较产生更高 token 和重试
    {"trace_id": "T-805", "request": "取消昨晚提交的酒店预订", "prompt_tokens": 390, "completion_tokens": 60, "tool": "hotel_cancel", "tool_ms": 300, "tool_cost": 0.02, "retries": 1, "status": "error"},  # 取消工具最终超时失败
]  # 结束五条脱敏 Agent 运行
print("运行输入：trace | status | retries | tool | user_request")  # 展示可观测系统接收的业务维度
for run in runs:  # 逐条输出五个差旅请求
    print(f"{run['trace_id']} | {run['status']:5} | {run['retries']} | {run['tool']:13} | {run['request']}")  # 让正常、重试和失败运行同时可见


运行输入：trace | status | retries | tool | user_request
T-801 | ok    | 0 | train_search  | 查询明早上海到杭州高铁
T-802 | ok    | 1 | hotel_search  | 预订北京两晚 600 元内酒店
T-803 | ok    | 0 | ticket_change | 把 G12 改签到下午
T-804 | ok    | 2 | multi_search  | 比较飞机和高铁总预算
T-805 | error | 1 | hotel_cancel  | 取消昨晚提交的酒店预订


## Baseline / 基线：无身份的文本日志

字符串日志可以被人搜索，却没有 trace_id、parent_id 或稳定 span 身份。两个并发请求都输出“tool finished”时，无法知道费用和失败属于谁。

In [2]:
plain_logs = ["received user request", "planning completed", "tool finished in 360ms", "received user request", "tool timeout", "answer sent"]  # 构造两个运行交错后的无身份文本日志
tool_lines = [line for line in plain_logs if "tool" in line]  # 尝试从文本中提取工具相关日志
attributable_lines = [line for line in tool_lines if "T-" in line]  # 检查是否能归因到具体 trace
print("交错文本日志：")  # 输出无结构基线供人工观察
for line in plain_logs:  # 逐行展示无法关联的日志
    print(line)  # 保留原始文本顺序
print(f"工具日志={len(tool_lines)}，可归因到请求={len(attributable_lines)}")  # 量化基线归因能力缺失


交错文本日志：
received user request
planning completed
tool finished in 360ms
received user request
tool timeout
answer sent
工具日志=2，可归因到请求=0


## 核心实现：结构化 Span、执行树与成本归因

每条运行生成 root、plan、tool 和 answer 四个 Span。模型成本按教学单价计算，工具费用单独记录；parent_id 使执行树可重放。

In [3]:
prompt_rate = 0.000002  # 定义每个输入 token 的教学成本单价
completion_rate = 0.000006  # 定义每个输出 token 的教学成本单价
spans = []  # 收集五条运行的结构化 Span
for index, run in enumerate(runs):  # 为每个差旅请求生成确定性执行轨迹
    base = index * 2000  # 为不同 trace 分配不重叠的教学时间轴
    root_id = f"{run['trace_id']}:root"  # 创建当前请求的根 Span 身份
    plan_id = f"{run['trace_id']}:plan"  # 创建 LLM 规划 Span 身份
    tool_id = f"{run['trace_id']}:tool"  # 创建工具调用 Span 身份
    answer_id = f"{run['trace_id']}:answer"  # 创建最终回答 Span 身份
    plan_end = base + 80 + run["prompt_tokens"] // 10  # 用 token 数构造确定性规划时延
    tool_end = plan_end + run["tool_ms"] * (1 + run["retries"])  # 将重试次数纳入工具结束时刻
    answer_end = tool_end + 40 + run["completion_tokens"] // 5  # 用输出 token 构造回答时延
    model_cost = run["prompt_tokens"] * prompt_rate + run["completion_tokens"] * completion_rate  # 计算当前请求的模型教学成本
    spans.append({"trace_id": run["trace_id"], "span_id": root_id, "parent_id": None, "name": "agent_run", "start_ms": base, "end_ms": answer_end, "status": run["status"], "tokens": 0, "cost": 0.0, "retries": 0})  # 写入覆盖整次请求的根 Span
    spans.append({"trace_id": run["trace_id"], "span_id": plan_id, "parent_id": root_id, "name": "llm_plan", "start_ms": base + 10, "end_ms": plan_end, "status": "ok", "tokens": run["prompt_tokens"] // 2, "cost": model_cost * 0.45, "retries": 0})  # 写入规划阶段模型消耗
    spans.append({"trace_id": run["trace_id"], "span_id": tool_id, "parent_id": root_id, "name": run["tool"], "start_ms": plan_end, "end_ms": tool_end, "status": run["status"], "tokens": 0, "cost": run["tool_cost"], "retries": run["retries"]})  # 写入工具时延、费用和重试
    spans.append({"trace_id": run["trace_id"], "span_id": answer_id, "parent_id": root_id, "name": "llm_answer", "start_ms": tool_end, "end_ms": answer_end, "status": "skipped" if run["status"] == "error" else "ok", "tokens": run["prompt_tokens"] - run["prompt_tokens"] // 2 + run["completion_tokens"], "cost": model_cost * 0.55, "retries": 0})  # 写入回答阶段及失败跳过状态
focus_spans = [span for span in spans if span["trace_id"] == "T-802"]  # 选择含一次重试的酒店请求展示执行树
print("T-802 Trace：span | parent | name | start-end | status | tokens | cost | retries")  # 输出结构化轨迹的关键字段
for span in focus_spans:  # 逐 Span 展示根、规划、工具和回答
    print(f"{span['span_id']} | {span['parent_id']} | {span['name']} | {span['start_ms']}-{span['end_ms']} | {span['status']} | {span['tokens']} | {span['cost']:.4f} | {span['retries']}")  # 显示父子关系和成本来源


T-802 Trace：span | parent | name | start-end | status | tokens | cost | retries
T-802:root | None | agent_run | 2000-2936 | ok | 0 | 0.0000 | 0
T-802:plan | T-802:root | llm_plan | 2010-2148 | ok | 340 | 0.0010 | 0
T-802:tool | T-802:root | hotel_search | 2148-2868 | ok | 0 | 0.0300 | 1
T-802:answer | T-802:root | llm_answer | 2868-2936 | ok | 480 | 0.0012 | 0


## 失败案例与修正：孤儿 Span 与重复身份破坏重放

网络重试可能重复上报同一个 span_id，错误埋点也可能引用不存在的 parent_id。重放前先验证身份唯一、父节点存在、时间窗口合法；失败轨迹隔离而不是静默拼树。

In [4]:
orphan = {"trace_id": "T-802", "span_id": "T-802:orphan", "parent_id": "T-802:missing", "name": "late_callback", "start_ms": 2600, "end_ms": 2610, "status": "ok", "tokens": 0, "cost": 0.0, "retries": 0}  # 构造引用不存在父节点的孤儿 Span
duplicate = dict(focus_spans[1])  # 复制已有 plan Span 制造重复身份
broken_spans = focus_spans + [orphan, duplicate]  # 组合正常与损坏轨迹
def validate_trace(trace_spans):  # 在重放和指标计算前验证结构化轨迹
    ids = [span["span_id"] for span in trace_spans]  # 收集当前 trace 的所有 Span 身份
    id_set = set(ids)  # 建立父节点存在性索引
    errors = []  # 收集重复、孤儿和时间边界错误
    for span_id in sorted(set(ids)):  # 逐个检查 Span 身份出现次数
        if ids.count(span_id) > 1:  # 同一身份重复会让响应和成本重复计算
            errors.append((span_id, "duplicate_span"))  # 记录重复身份错误
    for span in trace_spans:  # 检查每个 Span 的父关系和时间边界
        if span["parent_id"] is not None and span["parent_id"] not in id_set:  # 非根节点必须引用当前 trace 内父节点
            errors.append((span["span_id"], "orphan_parent"))  # 记录孤儿节点错误
        if span["end_ms"] < span["start_ms"]:  # 结束时间不能早于开始时间
            errors.append((span["span_id"], "negative_duration"))  # 记录无效时间窗口
    return errors  # 返回可用于隔离和告警的错误列表
trace_errors = validate_trace(broken_spans)  # 验证包含孤儿和重复身份的损坏轨迹
repaired_spans = focus_spans  # 修正方案使用原始唯一且父关系完整的 Span 集
repaired_errors = validate_trace(repaired_spans)  # 重新验证修复后的轨迹
print("损坏轨迹错误：", trace_errors)  # 展示两个结构错误的具体身份和原因
print("修复后错误：", repaired_errors)  # 展示合法轨迹可以安全进入重放


损坏轨迹错误： [('T-802:plan', 'duplicate_span'), ('T-802:orphan', 'orphan_parent')]
修复后错误： []


## 结果表：每请求延迟、token、成本和失败点

In [5]:
metrics = []  # 收集五条 trace 的请求级指标
print("trace | latency_ms | tokens | cost | retries | status | failed_span")  # 输出成本和故障归因表
for run in runs:  # 对每个请求独立聚合结构化 Span
    trace_spans = [span for span in spans if span["trace_id"] == run["trace_id"]]  # 获取当前 trace 的四个 Span
    root = next(span for span in trace_spans if span["parent_id"] is None)  # 找到根 Span 计算端到端时延
    latency = root["end_ms"] - root["start_ms"]  # 计算当前请求端到端毫秒数
    tokens = sum(span["tokens"] for span in trace_spans)  # 汇总规划和回答 token
    cost = sum(span["cost"] for span in trace_spans)  # 汇总模型与工具教学成本
    retries = sum(span["retries"] for span in trace_spans)  # 汇总工具重试次数
    failed_span = next((span["name"] for span in trace_spans if span["status"] == "error"), "-")  # 定位第一个失败步骤
    metrics.append({"trace_id": run["trace_id"], "latency": latency, "tokens": tokens, "cost": cost, "retries": retries, "status": run["status"], "failed_span": failed_span})  # 保存请求级聚合指标
    print(f"{run['trace_id']} | {latency:4} | {tokens:4} | {cost:.4f} | {retries} | {run['status']:5} | {failed_span}")  # 逐条展示成本和失败归因
total_cost = sum(metric["cost"] for metric in metrics)  # 计算五条教学运行总成本
slowest_trace = max(metrics, key=lambda metric: metric["latency"])["trace_id"]  # 找出端到端时延最高的请求
failed_traces = [metric["trace_id"] for metric in metrics if metric["status"] == "error"]  # 汇总失败请求身份
print(f"汇总：total_cost={total_cost:.4f}，slowest={slowest_trace}，failed={failed_traces}")  # 输出可用于运营决策的顶层指标


trace | latency_ms | tokens | cost | retries | status | failed_span
T-801 |  361 |  515 | 0.0114 | 0 | ok    | -
T-802 |  936 |  820 | 0.0322 | 1 | ok    | -
T-803 |  435 |  630 | 0.0517 | 0 | ok    | -
T-804 | 1784 | 1130 | 0.0831 | 2 | ok    | -
T-805 |  771 |  450 | 0.0211 | 1 | error | agent_run
汇总：total_cost=0.1996，slowest=T-804，failed=['T-805']


## 结果解读

T-802 的工具 Span 因一次重试占据主要时延，且费用可以分别归因到 plan、hotel_search 和 answer。T-804 有最多 token 和两次重试，因此成为最慢请求；T-805 的失败点明确定位为 hotel_cancel。无身份文本日志只能看到一次 timeout，无法回答它属于谁、花了多少钱或是否影响最终回答。

## 生产边界

生产可观测性需要跨服务 trace context、采样策略、时钟偏差处理、敏感输入摘要、指标基数控制、存储保留和成本价格版本。Trace 重放只能复现控制流，若工具外部状态已变化，必须使用录制响应或只读模拟器，不能重新产生副作用。本例没有模拟异步并行 Span 和真实模型流式 token。

## 最小回归测试

In [6]:
assert len(runs) >= 5  # 保证案例覆盖正常、重试和失败的多条运行
assert len(spans) == len(runs) * 4  # 保证每条请求都有根、规划、工具和回答 Span
assert len(attributable_lines) == 0  # 保证无身份日志基线真实无法归因
assert ("T-802:plan", "duplicate_span") in trace_errors  # 保证重复 Span 在重放前被发现
assert ("T-802:orphan", "orphan_parent") in trace_errors  # 保证孤儿父关系在重放前被发现
assert repaired_errors == []  # 保证修复后的 T-802 轨迹结构完整
assert failed_traces == ["T-805"]  # 保证失败运行可以准确归因到唯一 trace
